# Thử scoring Controller v4 trên Tesla T4 — setup từ đầu

Chỉnh một cell CONFIG rồi **Runtime → Run all**. Chỉ cần Secret `GITHUB_TOKEN`; repo/ref đã có mặc định. Không cần giữ checkpoint/model/manifest cũ.

Lần đầu notebook tải UDIS-D training ZIP, tự chuẩn bị 200 pair calibration + 50 pair validation, build/test, sweep 48 cấu hình, thu feature và huấn luyện một baseline v4 photo mới. Sau đó khóa model và chạy Reference + P0–P5. Lần sau dùng model/data đã lưu trên Drive, không huấn luyện giữa các policy.

Đây là **baseline dựng lại**, không tái lập đúng weights hay số đo v4 đã mất. Không dùng tập test và không tuyên bố đánh giá độc lập từ cohort có thể trùng lịch sử. Cần source mới đã push trên GitHub; agent không tự push.


In [ ]:
# CONFIG — the only cell to edit. Never put a token here.
CONFIG = {
    "REPOSITORY": "nh-atuan/crossstage-stitch",
    "GIT_REF": "de/optimize-controller",  # Must contain score_setup.py on GitHub.
    "MOUNT_DRIVE": True,
    "MODE": "FULL",  # First-run SMOKE checks Reference only; FULL builds and freezes the model.
    "WORK_ROOT": "/content/drive/MyDrive/controller-score-work",
    "OUTPUT_ROOT": "/content/drive/MyDrive/controller-score-results",
    "TRAINING_ZIP": "",  # Optional private Drive path: /content/drive/MyDrive/UDIS/training.zip
    "TRAINING_DRIVE_ID": "1CTKa42BRdXO2vdYSadaQkvgQ8DR6l49a",  # Official training.zip; used if path above is empty.
    "SETUP_BUNDLE_ZIP": "",  # Optional: restore the setup bundle exported by an earlier run.
}


## 1. Kiểm GPU và lưu dữ liệu

Runtime phải có đúng một Tesla T4 và nvcc. Mount Drive để checkpoint còn sau khi mất runtime. Nếu dùng ổ `/content`, cần tải ZIP trước khi phiên kết thúc. Mọi lượt đo dùng seed `20261101`, hai warmup và mười repeat.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import uuid

if CONFIG["MODE"] not in ("SMOKE", "FULL"):
    raise RuntimeError("CONFIG.MODE phải là SMOKE hoặc FULL")
try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,uuid", "--format=csv,noheader"], text=True).strip().splitlines()
    if len(gpu) != 1 or "Tesla T4" not in gpu[0]:
        raise RuntimeError("Cần đúng một GPU Tesla T4; chọn lại runtime T4 rồi Run all.")
    subprocess.run(["nvcc", "--version"], check=True)
except FileNotFoundError:
    raise RuntimeError("Runtime chưa có GPU/CUDA. Chọn GPU Tesla T4 rồi Run all.") from None
print("GPU:", gpu[0], "SEED: 20261101; WARMUP: 2; REPEATS: 10")
if CONFIG["MOUNT_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
# Ephemeral /content identity survives reruns in this runtime, not a replacement VM.
_session_path = Path("/content/controller-score-runtime-id")
if not _session_path.exists():
    _session_path.write_text(uuid.uuid4().hex)
CONFIG["SESSION_ID"] = _session_path.read_text().strip()


## 2. Clone private repo và khóa commit

Trong Colab, mở biểu tượng chìa khóa **Secrets**, tạo `GITHUB_TOKEN` và bật **Notebook access**. Token Fine-grained chỉ cần quyền **Contents: Read-only** cho đúng repo. Cell không in token, không ghi token vào URL hoặc Git config. Token chỉ được truyền tới tiến trình Git qua bộ nhớ môi trường con và ASKPASS tạm; file ASKPASS chỉ chứa code đọc biến môi trường.

Mỗi Run all fetch lại branch rồi khóa full commit SHA cho lượt đó. Có thể nhận bản sửa mới ngay trong cùng runtime; không xóa dữ liệu Drive. Nếu GIT_REF là full SHA đã checkout thì dùng lại bản đã khóa.


In [ ]:
import re
import shutil
import tempfile


def checkout_private(repository, ref, workspace, get_secret):
    if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository) or repository == "OWNER/REPO":
        raise RuntimeError("Điền CONFIG.REPOSITORY theo dạng OWNER/REPO.")
    if not ref or ref.startswith("-") or any(c in ref for c in "\r\n"):
        raise RuntimeError("Điền CONFIG.GIT_REF bằng branch hoặc commit hợp lệ đã push.")
    checkout = Path(workspace) / "controller-score-source"
    lock = checkout / ".git/score-checkout.json"
    existing = lock.exists()
    if existing:
        stored = json.loads(lock.read_text())
        actual = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(checkout), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
        if stored["repository"] != repository or stored["revision"] != actual or dirty:
            raise RuntimeError("Checkout đã đổi repo/HEAD hoặc có sửa tracked files. Giữ checkpoint trên Drive và dùng checkout sạch; không ghi đè thay đổi.")
        if re.fullmatch(r"[0-9a-f]{40}", ref) and actual == ref:
            return checkout, actual
    elif checkout.exists():
        raise RuntimeError("Checkout dở dang. Xóa riêng /content/controller-score-source trong Files rồi Run all; giữ checkpoint trên Drive.")
    try:
        token = get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Không đọc được GITHUB_TOKEN. Mở Secrets, thêm token và bật Notebook access; kiểm quyền repository.") from None
    if not isinstance(token, str) or not token.strip():
        token = None
        raise RuntimeError("GITHUB_TOKEN trống. Tạo Fine-grained PAT Contents: Read-only cho đúng repo trong Colab Secrets.")
    child_env = {k:v for k,v in os.environ.items() if not k.startswith("GIT_") and k not in ("GITHUB_TOKEN", "GH_TOKEN")}
    try:
        with tempfile.TemporaryDirectory(prefix="score-git-auth-") as auth:
            askpass = Path(auth) / "askpass.py"
            askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['SCORE_GITHUB_TOKEN'])\n")
            askpass.chmod(0o700)
            child_env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0", SCORE_GITHUB_TOKEN=token)
            args = ["git", "-c", "credential.helper=", "-c", "core.askPass="+str(askpass), "-c", "http.extraHeader="]
            commands = [] if existing else [args+["init", str(checkout)],
                args+["-C", str(checkout), "remote", "add", "origin", "https://github.com/"+repository+".git"]]
            commands += [args+["-C", str(checkout), "fetch", "--depth=1", "origin", ref],
                args+["-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"]]
            for command in commands:
                completed = subprocess.run(command, env=child_env, capture_output=True, text=True)
                if completed.returncode:
                    # Suppress all network output; no credential-bearing output reaches a cell/log.
                    raise RuntimeError("Clone/checkout thất bại. Kiểm OWNER/REPO, ref đã push, Contents: Read-only và approval của tổ chức; không đổi sang token ghi hoặc URL chứa token.")
    finally:
        token = None
        child_env.pop("SCORE_GITHUB_TOKEN", None)
        child_env.pop("GIT_ASKPASS", None)
    revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise RuntimeError("Không khóa được full commit SHA")
    lock.write_text(json.dumps({"repository": repository, "ref": ref, "revision": revision}))
    return checkout, revision


from google.colab import userdata
SOURCE, REVISION = checkout_private(CONFIG["REPOSITORY"], CONFIG["GIT_REF"], "/content", userdata.get)
os.chdir(SOURCE)
if not (SOURCE/"tools/controller/score_setup.py").is_file():
    raise RuntimeError("Ref/checkout chưa chứa setup mới. Push commit đã bàn giao; nếu runtime đang khóa checkout cũ, mở runtime mới rồi Run all.")
print("LOCKED COMMIT:", REVISION)


## 3. Cài dependencies

Không cần upload model hay tự tạo manifest. Mặc định tự tải `training.zip` từ File ID tác giả. Nếu Google Drive báo quota, tải [training.zip](https://drive.google.com/file/d/1CTKa42BRdXO2vdYSadaQkvgQ8DR6l49a/view) bằng trình duyệt, upload nguyên ZIP vào MyDrive và điền `TRAINING_ZIP`; không giải nén. Có thể thay `TRAINING_DRIVE_ID` bằng ID bản sao công khai; file riêng tư dùng đường dẫn Drive đã mount.

Model dựng lại dùng pipeline XGBoost photo hiện có. P0–P5 giữ nguyên model, feature và post-gate 0.65; chỉ thay score/pre-threshold. Đây là ngoại lệ huấn luyện một lần đã được Đế chấp thuận vì checkpoint cũ bị mất.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake", "nbformat", "gdown",
    "-r", "tools/controller/requirements-learn.txt", "-r", "tools/controller/requirements-report.txt"], check=True)
# Re-import repository tooling after a branch refresh in the same runtime.
import importlib
for _name in list(sys.modules):
    if _name == "tools" or _name.startswith("tools."):
        del sys.modules[_name]
importlib.invalidate_caches()
sys.path.insert(0, str(SOURCE))
print("SETUP: tự tải/đọc training.zip, chia split, chuyển PPM; model đã lưu sẽ được dùng lại.")


## 4. Tự setup, khóa baseline và benchmark

Lần đầu FULL: kiểm ZIP training thật; khóa split theo component ảnh với seed 20261101; chỉ decode 250 pair được chọn sang P6 PPM; build Release `sm_75`; CTest/pytest; sweep calibration 48 cấu hình với 2 warmup/10 repeat; thu feature riêng và đối chiếu H; huấn luyện v4 photo bằng code hiện có. Baseline xuất với pre 0.85, gate 0.65, extra_ms 0.429087 trước khi mở validation. Giữ cả export gốc của trainer và nhãn reconstructed.

Model/data được lưu tại WORK_ROOT và đóng `controller_v4_reconstructed_setup_bundle.zip`. Nếu mất runtime sau khi setup xong, Run all dùng lại model; nếu mới có partial sweep thì runtime mới mở campaign setup riêng, không trộn các GPU session. Không tự thay pair khó hoặc dùng synthetic fallback.

P0–P2 dùng score cũ, P3–P5 dùng reuse approximation; mỗi nhóm có pre-threshold 0.85/0.70/0.50. Profile Reference trên calibration dùng binary hiện tại; R-only trừ S1–S4, đổi K trừ S1, đổi S không giảm. Giữ full S5, S6 và overhead. `1-p_pre` là failure proxy; score chưa phải expected E2E chính xác. Phân tích dựa trên E2E đo thật và canonical QoR.

Nếu MODE=SMOKE và chưa có model, chỉ kiểm Reference trên 3 pair, không fit model hoặc giả lập sáu policy. FULL tự smoke sau khi khóa model rồi chạy cả cohort, không cần chọn policy hay nhập lệnh. Oracle calibration không được dùng làm Oracle validation; chưa có sweep validation phù hợp thì báo UNAVAILABLE, không tự chạy sweep thêm.


In [ ]:
from tools.controller.score_setup import automatic
RESULT_ZIP = automatic(CONFIG)


## 5. Tải ZIP và gửi lại

Cell cuối tải ZIP kết quả; gửi ZIP cùng LOCKED COMMIT trong chat. ZIP có CSV/JSON, raw runs, traces, model sáu policy, diagnostics, hashes và bằng chứng dựng baseline. Giữ `controller_v4_reconstructed_setup_bundle.zip` trong WORK_ROOT trên Drive; file này chứa model, split và ảnh PPM, dùng để khôi phục lần sau qua SETUP_BUNDLE_ZIP.

Nếu setup lỗi, log và ZIP partial nằm trong WORK_ROOT/setup-sessions. Khi mất runtime, dữ liệu trên Drive vẫn còn; ổ /content không có bảo đảm đó. Không đổi default policy hoặc chọn ngưỡng thắng rồi gọi validation đã xem là đánh giá độc lập.


In [ ]:
from google.colab import files
print("Gửi ZIP:", RESULT_ZIP)
print("Git commit:", REVISION)
files.download(str(RESULT_ZIP))
